In [ ]:
def create_transformer_model_sparse(input_vocab_size, target_vocab_size, embedding_dim, 
                                   num_heads, ff_dim, num_layers, max_seq_length, dropout_rate=0.1):
    # 1. Encoder setup
    encoder_inputs = Input(shape=(max_seq_length,))
    encoder_embedding = Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
    
    # 2. Positional encoding
    encoder_pos_enc = PositionalEncoding(max_seq_length, embedding_dim)
    decoder_pos_enc = PositionalEncoding(max_seq_length, embedding_dim)
    
    encoder_embedding = encoder_pos_enc(encoder_embedding)
    encoder_embedding = Dropout(dropout_rate)(encoder_embedding)
    
    # 3. Encoder stack
    encoder_outputs = encoder_embedding
    for _ in range(num_layers):
        encoder_outputs = TransformerEncoder(
            intermediate_dim=ff_dim, num_heads=num_heads
        )(encoder_outputs)
        encoder_outputs = Dropout(dropout_rate)(encoder_outputs)
    
    # 4. Decoder setup
    decoder_inputs = Input(shape=(max_seq_length,))
    decoder_embedding = Embedding(target_vocab_size, embedding_dim)(decoder_inputs)
    decoder_embedding = decoder_pos_enc(decoder_embedding)
    decoder_embedding = Dropout(dropout_rate)(decoder_embedding)
    
    # 5. Decoder stack
    decoder_outputs = decoder_embedding
    for _ in range(num_layers):
        decoder_outputs = TransformerDecoder(
            intermediate_dim=ff_dim, num_heads=num_heads
        )(decoder_outputs, encoder_outputs)
        decoder_outputs = Dropout(dropout_rate)(decoder_outputs)
    
    # 6. Final output
    decoder_outputs = Dense(target_vocab_size, activation='softmax')(decoder_outputs)
    
    # 7. Model instantiation
    model = Model(inputs=[encoder_inputs, decoder_inputs], outputs=decoder_outputs)
    
    return model


def translate_sequence(input_sequence, model, arabic_tokenizer, latin_tokenizer, max_seq_length):
    # 1. Preprocess input
    input_seq = arabic_tokenizer.texts_to_sequences([input_sequence])
    padded_input_seq = pad_sequences(input_seq, maxlen=max_seq_length, padding="post")
    
    # 2. Initialize target
    translated_sequence = []
    target_seq = np.zeros((1, max_seq_length), dtype='int32')
    
    # 3. Setup special tokens
    start_token = latin_tokenizer.word_index.get('#', 1)
    end_token = latin_tokenizer.word_index.get('$', 2)
    target_seq[0, 0] = start_token
    
    # 4. Generation loop
    for i in range(1, max_seq_length):
        # A. Predict
        output = model.predict([padded_input_seq, target_seq], verbose=0)
        
        # B. Select token
        predicted_token_index = np.argmax(output[0, i-1, :])
        
        # C. Check stopping
        if predicted_token_index == end_token or predicted_token_index == 0:
            break
        
        # D. Update
        translated_sequence.append(predicted_token_index)
        if i < max_seq_length:
            target_seq[0, i] = predicted_token_index
    
    # 5. Decode to text
    translated_text = latin_tokenizer.sequences_to_texts([translated_sequence])[0]
    translated_text = translated_text.replace(" ", "")
    
    return translated_text

# Exercise 3: Sequence-to-Sequence Models for Transliteration (Arabic to English)

## Introduction

In this exercise, we will explore the Transformer encoder-decoder model for the task of **transliteration of Arabic words to English**. Transliteration is the process of converting text from one script to another, and in this case, we aim to convert Arabic script into Latin script using sequence-to-sequence models. The Transformer model, which uses self-attention mechanisms, will be implemented to learn the relationship between Arabic and English characters for transliteration. We will evaluate the performance of the model using **BLEU scores**.

---

## Tasks

### Task 1: Dataset Analysis and Preprocessing

1. **Analyze the dataset**: Inspect the distribution of sequence lengths and choose appropriate maximum sequence lengths for both the input and output sequences.
   
2. **Choose vocabulary sizes**: Decide on the vocabulary sizes for both input (Arabic characters) and output (Latin characters). Consider using character-level tokenization.
   
3. **Sequence Padding and Truncation**: Pad or truncate sequences to the chosen maximum length and prepare input-output pairs for training.

---

### Task 2: Transformer Encoder-Decoder Model

1. **Implement the Transformer encoder-decoder model**:
   - Use the encoder-decoder architecture with multi-head self-attention.
   - The encoder will process the input Arabic sequence, and the decoder will generate the output transliterated English sequence.
   - Use **KerasNLP** to build the model. Below are additional hints:
     - Define separate embedding layers for the Arabic input and English output.
     - Use positional encodings to account for the order of characters in sequences.
     - Use a `keras.layers.LayerNormalization` layer to normalize inputs to the encoder and decoder.
     - Add dropout layers for regularization.

2. **Train the Transformer model** on the preprocessed transliteration dataset.

3. **Evaluate the model** by generating transliterated sequences for sample Arabic inputs.
   

---

### Task 3: BLEU Score Evaluation

1. **Use BLEU score** to evaluate the performance of the Transformer encoder-decoder model on the transliteration task:
   - Use the `nltk.translate.bleu_score` module for BLEU score computation.
   - Calculate BLEU scores for unigram, bigram, trigram, and 4-gram precision.

2. **Compare the BLEU scores** to determine the quality of the transliterated outputs and assess how well the model performs.

---

### Additional Hints for Implementation

#### Input and Output Preparation
- Use the **`tf.keras.preprocessing.text.Tokenizer`** for character-level tokenization.
- Ensure that both input and output sequences include special tokens:
  - `<start>` and `<end>` tokens for the output sequence.
  - Pad sequences with `<pad>` tokens to the maximum length.

#### Model Definition
- Implement the encoder as a stack of multi-head self-attention layers followed by feed-forward layers.
- Implement the decoder similarly but include cross-attention layers that attend to the encoder output.

#### Training and Evaluation
- Use teacher forcing during training to improve convergence.
- For inference, feed the predicted token from the previous time step as input to the decoder at the next time step.
- Evaluate on a held-out test set and analyze failure cases.

By following these steps and hints, you will be able to successfully implement and evaluate a Transformer encoder-decoder model for Arabic to English transliteration.


In [ ]:
!pip install nltk
!pip install keras-nlp
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
!apt-get install unrar
!unrar x EN-AR_Translit.rar
!ls

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

def load_transliteration_dataset(base_path):
    """
    Load transliteration dataset from text files.

    Args:
        base_path (str): Path to the directory containing dataset files

    Returns:
        dict: Dictionary containing loaded datasets
    """
    dataset_files = {
        'train_ar': os.path.join(base_path, 'EN-AR_Translit/train.ar'),
        'train_en': os.path.join(base_path, 'EN-AR_Translit/train.en'),
        'test_ar': os.path.join(base_path, 'EN-AR_Translit/test.ar'),
        'test_en': os.path.join(base_path, 'EN-AR_Translit/test.en'),
        'dev_ar': os.path.join(base_path, 'EN-AR_Translit/dev.ar'),
        'dev_en': os.path.join(base_path, 'EN-AR_Translit/dev.en')
    }

    loaded_data = {}
    for key, filepath in dataset_files.items():
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                loaded_data[key] = ["#" + line.strip() + "$" for line in f.readlines()]
        except FileNotFoundError:
            print(f"Warning: File {filepath} not found.")
            loaded_data[key] = []

    return loaded_data


loaded_data = load_transliteration_dataset(base_path="")
print(len(loaded_data["train_ar"]),len(loaded_data["train_en"]))
print(len(loaded_data["dev_ar"]),len(loaded_data["dev_en"]))
print(len(loaded_data["test_ar"]),len(loaded_data["test_en"]))

print(loaded_data["dev_ar"][0],loaded_data["dev_en"][0])

75907 75907
1003 1003
3014 3014
اميس Amis


In [ ]:

def analyze_dataset_statistics(dataset):
    """
    Analyze dataset statistics and visualize sequence lengths.

    Args:
        dataset (dict): Loaded dataset dictionary
    """
    # Compute sequence lengths
    length_analysis = {
        'train_ar_lengths': [len(word) for word in dataset['train_ar']],
        'train_en_lengths': [len(word) for word in dataset['train_en']],
        'dev_ar_lengths': [len(word) for word in dataset['dev_ar']],
        'dev_en_lengths': [len(word) for word in dataset['dev_en']],
        'test_ar_lengths': [len(word) for word in dataset['test_ar']],
        'test_en_lengths': [len(word) for word in dataset['test_en']]
    }

    # Print dataset sizes
    print("Dataset Sizes:")
    print(f"Train: {len(dataset['train_ar'])} instances")
    print(f"Dev: {len(dataset['dev_ar'])} instances")
    print(f"Test: {len(dataset['test_ar'])} instances")

    # Print length statistics
    print("\nSequence Length Statistics:")
    for name, lengths in length_analysis.items():
        print(f"{name}:")
        print(f"  Mean length: {np.mean(lengths):.2f}")
        print(f"  Max length: {np.max(lengths)}")
        print(f"  Min length: {np.min(lengths)}")

    # Visualize length distributions
    plt.figure(figsize=(15, 5))

    # Arabic lengths subplot
    plt.subplot(1, 2, 1)
    sns.boxplot(data=[
        length_analysis['train_ar_lengths'],
        length_analysis['dev_ar_lengths'],
        length_analysis['test_ar_lengths']
    ])
    plt.title('Arabic Sequence Lengths')
    plt.xticks([0, 1, 2], ['Train', 'Dev', 'Test'])
    plt.ylabel('Sequence Length')

    # English lengths subplot
    plt.subplot(1, 2, 2)
    sns.boxplot(data=[
        length_analysis['train_en_lengths'],
        length_analysis['dev_en_lengths'],
        length_analysis['test_en_lengths']
    ])
    plt.title('English Sequence Lengths')
    plt.xticks([0, 1, 2], ['Train', 'Dev', 'Test'])
    plt.ylabel('Sequence Length')

    plt.tight_layout()
    plt.show()

    return length_analysis

length_analysis = analyze_dataset_statistics(loaded_data)

In [ ]:

def fit_tokenizers(arabic_words, latin_words):
    """
    Fit tokenizers on the Arabic and Latin words.

    Args:
        arabic_words (list): List of Arabic words
        latin_words (list): List of Latin transliterations

    Returns:
        tuple: Fitted Arabic and Latin tokenizers
    """
    # Character-level tokenizers
    arabic_tokenizer = Tokenizer(char_level=True)
    latin_tokenizer = Tokenizer(char_level=True)

    # Fit tokenizers
    arabic_tokenizer.fit_on_texts(arabic_words)
    latin_tokenizer.fit_on_texts(latin_words)

    return arabic_tokenizer, latin_tokenizer


def transform_data(arabic_words, latin_words, arabic_tokenizer, latin_tokenizer, max_seq_length):
    """
    Transform the transliteration data for a seq2seq model with separate encoder and decoder inputs.

    Note: The dataset loading already adds # (start) and $ (end) tokens to both Arabic and Latin words.
    - Encoder input: keeps both # and $ tokens
    - Decoder input: keeps only # (start token), removes $
    - Decoder target: keeps only $ (end token), removes #
    """

    # Prepare encoder input (Arabic with both start and end tokens - no changes needed)
    encoder_input = arabic_words  # Already has #word$
    encoder_input_sequences = arabic_tokenizer.texts_to_sequences(encoder_input)
    encoder_input_padded = pad_sequences(encoder_input_sequences, maxlen=max_seq_length, padding='post', truncating='post')

    # Prepare decoder input (Latin with start token only - remove end token $)
    decoder_input = [word[:-1] if word.endswith("$") else word for word in latin_words]  # Remove $ (end token)
    decoder_input_sequences = latin_tokenizer.texts_to_sequences(decoder_input)
    decoder_input_padded = pad_sequences(decoder_input_sequences, maxlen=max_seq_length, padding='post', truncating='post')

    # Prepare decoder target (Latin with end token only - remove start token #)
    decoder_target = [word[1:] if word.startswith("#") else word for word in latin_words]  # Remove # (start token)
    decoder_target_sequences = latin_tokenizer.texts_to_sequences(decoder_target)
    decoder_target_padded = pad_sequences(decoder_target_sequences, maxlen=max_seq_length, padding='post', truncating='post')

    return encoder_input_padded, decoder_input_padded, decoder_target_padded


max_seq_length = 20
arabic_tokenizer, latin_tokenizer = fit_tokenizers(loaded_data["train_ar"], loaded_data["train_en"])
train_encoder_input, train_decoder_input, train_decoder_target = transform_data(loaded_data["train_ar"], loaded_data["train_en"], arabic_tokenizer, latin_tokenizer, max_seq_length=max_seq_length)
dev_encoder_input, dev_decoder_input, dev_decoder_target = transform_data(loaded_data["dev_ar"], loaded_data["dev_en"], arabic_tokenizer, latin_tokenizer, max_seq_length=max_seq_length)
test_encoder_input, test_decoder_input, test_decoder_target = transform_data(loaded_data["test_ar"], loaded_data["test_en"], arabic_tokenizer, latin_tokenizer, max_seq_length=max_seq_length)



In [ ]:
print(len(arabic_tokenizer.word_index),len(latin_tokenizer.word_index))
print(arabic_tokenizer.word_index)
print(latin_tokenizer.word_index)
print(loaded_data["train_ar"][10],train_encoder_input[10])
print(loaded_data["train_en"][10],train_decoder_input[10])
print(loaded_data["train_en"][10],train_decoder_target[10])


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Embedding, Dense, LayerNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from keras_nlp.layers import TransformerEncoder, TransformerDecoder
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_seq_length, embedding_dim):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(max_seq_length, embedding_dim)

    def get_angles(self, position, i, embedding_dim):
        angles = 1 / tf.pow(10000.0, (2 * (i // 2)) / tf.cast(embedding_dim, tf.float32))
        return position * angles

    def positional_encoding(self, max_seq_length, embedding_dim):
        angle_rads = self.get_angles(
            position=tf.range(max_seq_length, dtype=tf.float32)[:, tf.newaxis],
            i=tf.range(embedding_dim, dtype=tf.float32)[tf.newaxis, :],
            embedding_dim=embedding_dim
        )
        # Apply sin to even indices
        sines = tf.math.sin(angle_rads[:, 0::2])
        # Apply cos to odd indices
        cosines = tf.math.cos(angle_rads[:, 1::2])

        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = pos_encoding[tf.newaxis, ...]
        return tf.cast(pos_encoding, tf.float32)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.pos_encoding[:, :seq_len, :]


def create_transformer_model_sparse(input_vocab_size, target_vocab_size, embedding_dim, num_heads, ff_dim, num_layers, max_seq_length, dropout_rate=0.1):
    """
    Create the Transformer model with dropout.

    This constructs a full Encoder-Decoder architecture.
    - The Encoder processes the input sequence (source).
    - The Decoder processes the target sequence and attends to the Encoder's output.
    """

    # ==============================================================================
    # 1. ENCODER SETUP
    # ==============================================================================


    # TODO: Define the Encoder Input.
    # - Instruction: Use 'layers.Input'. Shape should be (max_seq_length,).
    # - Assign to variable 'encoder_inputs'.

    # TODO: Create Encoder Embeddings.
    # - Instruction: Use 'layers.Embedding'.
    #   Input dim: 'input_vocab_size', Output dim: 'embedding_dim'.
    # - Instruction: Apply this layer to 'encoder_inputs'.
    # - Assign to variable 'encoder_embedding'.


    # ==============================================================================
    # 2. POSITIONAL ENCODING PREPARATION
    # ==============================================================================
    # We need separate positional encoding instances for the encoder and decoder
    # because they might track state or be learned differently depending on implementation.

    # TODO: Instantiate Positional Encoding layers.
    # - Instruction: Instantiate the 'PositionalEncoding' class (assumed to be a custom Layer).
    #   Pass 'max_seq_length' and 'embedding_dim'.
    # - Assign to 'encoder_pos_enc'.
    # - Repeat for 'decoder_pos_enc'.

    # TODO: Apply Positional Encoding to Encoder.
    # - Instruction: Pass 'encoder_embedding' through 'encoder_pos_enc'.
    # - Update the variable 'encoder_embedding'.

    # TODO: Apply Initial Dropout.
    # - Instruction: Apply 'layers.Dropout(dropout_rate)' to 'encoder_embedding'.
    # - Update the variable 'encoder_embedding'.


    # ==============================================================================
    # 3. ENCODER STACK (The Loop)
    # ==============================================================================
    # We stack N encoder layers. Each processes the output of the previous one.

    # TODO: Initialize the loop variable.
    # - Instruction: Set 'encoder_outputs' = 'encoder_embedding'.

    # TODO: Loop 'num_layers' times.
    # - Instruction: Create a for loop using 'range(num_layers)'.
    #   Inside the loop:
    #   1. Instantiate and Call 'TransformerEncoder':
    #      - Args: 'intermediate_dim=ff_dim', 'num_heads=num_heads'.
    #      - Input: Pass 'encoder_outputs' into this layer.
    #      - Update 'encoder_outputs' with the result.
    #   2. Apply Dropout:
    #      - Apply 'layers.Dropout(dropout_rate)' to 'encoder_outputs'.
    #      - Update 'encoder_outputs'.


    # ==============================================================================
    # 4. DECODER SETUP
    # ==============================================================================
    # TODO: Define the Decoder Input.
    # - Instruction: Use 'layers.Input'. Shape should be (max_seq_length,).
    # - Assign to variable 'decoder_inputs'.

    # TODO: Create Decoder Embeddings.
    # - Instruction: Use 'layers.Embedding'.
    #   Input dim: 'target_vocab_size', Output dim: 'embedding_dim'.
    # - Instruction: Apply this to 'decoder_inputs'.
    # - Assign to 'decoder_embedding'.

    # TODO: Apply Positional Encoding and Dropout to Decoder.
    # - Instruction: Pass 'decoder_embedding' through 'decoder_pos_enc'.
    # - Instruction: Pass the result through 'layers.Dropout(dropout_rate)'.
    # - Update 'decoder_embedding' with the final result.


    # ==============================================================================
    # 5. DECODER STACK (Cross-Attention)
    # ==============================================================================
    # The Decoder needs TWO inputs:
    # 1. The target sequence (so far).
    # 2. The 'encoder_outputs' (to understand the context of the source sentence).

    # TODO: Initialize the loop variable.
    # - Instruction: Set 'decoder_outputs' = 'decoder_embedding'.

    # TODO: Loop 'num_layers' times.
    # - Instruction: Create a for loop using 'range(num_layers)'.
    #   Inside the loop:
    #   1. Instantiate and Call 'TransformerDecoder':
    #      - Args: 'intermediate_dim=ff_dim', 'num_heads=num_heads'.
    #      - Inputs: This layer requires a call structure like: layer(input_A, input_B).
    #        Pass 'decoder_outputs' as the first argument (Query).
    #        Pass 'encoder_outputs' as the second argument (Key/Value).
    #      - Update 'decoder_outputs'.
    #   2. Apply Dropout:
    #      - Apply 'layers.Dropout(dropout_rate)' to 'decoder_outputs'.
    #      - Update 'decoder_outputs'.


    # ==============================================================================
    # 6. FINAL OUTPUT
    # ==============================================================================
    # TODO: Project to Vocabulary.
    # - Instruction: Create a 'layers.Dense' layer.
    #   Units: 'target_vocab_size'.
    #   Activation: 'softmax'.
    # - Instruction: Apply this to 'decoder_outputs'.
    # - Update 'decoder_outputs'.


    # ==============================================================================
    # 7. MODEL INSTANTIATION
    # ==============================================================================
    # TODO: Build the Keras Model.
    # - Instruction: Use 'models.Model'.
    # - Inputs: Pass a list containing both inputs -> [encoder_inputs, decoder_inputs].
    # - Outputs: Pass 'decoder_outputs'.
    # - Assign to 'model'.

    return model

def train_transformer_model(model, train_encoder_input, train_decoder_input,
        train_decoder_target, dev_encoder_input, dev_decoder_input, dev_decoder_target,
        batch_size, epochs, callbacks=None):
    """
    Train the Transformer model with optional callbacks.
    """
    history = model.fit(
        [train_encoder_input, train_decoder_input],
        train_decoder_target,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=([dev_encoder_input, dev_decoder_input], dev_decoder_target),
        callbacks=callbacks  # Add callbacks parameter
    )
    return history


def plot_training_history_sparse(history):
    """
    Plot the training history.
    """
    plt.figure(figsize=(12, 4))

    # Get the metric key name dynamically
    metric_keys = [k for k in history.history.keys() if 'accuracy' in k.lower() and not k.startswith('val_')]
    metric_name = metric_keys[0] if metric_keys else 'accuracy'
    val_metric_name = 'val_' + metric_name if not metric_name.startswith('val_') else metric_name

    # Plot training & validation accuracy values
    plt.subplot(1, 2, 1)
    if metric_name in history.history:
        plt.plot(history.history[metric_name], label='Train Accuracy', linewidth=2)
    if val_metric_name in history.history:
        plt.plot(history.history[val_metric_name], label='Validation Accuracy', linewidth=2)
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot training & validation loss values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Parameters
embedding_dim = 128  # Increased from 64
num_heads = 4        # Increased from 2
ff_dim = 128         # Increased from 64
num_layers = 4       # Increased from 1
batch_size = 32      # Keep the same
epochs = 30          # Increased from 4 (LSTM used 2 but more complex model)


# Step 1: Create and compile the Transformer model
# The model will be initialized with the given parameters, such as vocabulary size, embedding dimension, etc.
transformer_model = create_transformer_model_sparse(
    input_vocab_size=len(arabic_tokenizer.word_index)+1,
    target_vocab_size=len(latin_tokenizer.word_index)+1,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    max_seq_length=max_seq_length
)

# Add model summary
print("\nModel Architecture:")
print("="*70)
transformer_model.summary()
print("="*70)

# Step 2: Compile the model
# The model is compiled with the Adam optimizer, a small learning rate, sparse categorical cross-entropy loss,
# and accuracy as the evaluation metric.
# Add a custom masked loss function before compilation
def masked_loss(y_true, y_pred):
    """
    Masked loss function that ignores padding tokens (index 0)
    """
    mask = tf.math.not_equal(y_true, 0)
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, reduction='none')
    loss = loss_object(y_true, y_pred)
    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    """
    Masked accuracy that ignores padding tokens (index 0)
    """
    mask = tf.math.not_equal(y_true, 0)
    y_pred_class = tf.argmax(y_pred, axis=-1)
    y_true = tf.cast(y_true, y_pred_class.dtype)
    matches = tf.equal(y_true, y_pred_class)
    matches = matches & mask
    matches = tf.cast(matches, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(matches) / tf.reduce_sum(mask)

transformer_model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss=masked_loss,
    metrics=[masked_accuracy]
)

# Step 3: Train the model using the training data
# Use callbacks for learning rate reduction
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

# Then in training:
history = train_transformer_model(
    transformer_model,
    train_encoder_input, train_decoder_input, train_decoder_target,
    dev_encoder_input, dev_decoder_input, dev_decoder_target,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=[reduce_lr, early_stop]  # Add this parameter
)

# Step 4: Plot the training history to visualize the training and validation accuracy and loss
plot_training_history_sparse(history)


In [ ]:


# Step 5: Evaluate on test set
print("\n" + "="*70)
print("EVALUATING ON TEST SET")
print("="*70)

test_loss, test_accuracy = transformer_model.evaluate(
    [test_encoder_input, test_decoder_input],
    test_decoder_target,
    batch_size=batch_size,
    verbose=1
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")




In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def translate_sequence(input_sequence, model, arabic_tokenizer, latin_tokenizer, max_seq_length):
    """
    Translate a single sequence from Arabic to Latin using a trained transformer model.

    This function performs autoregressive generation:
    1. Encode the input.
    2. Start the decoder with a specific start token.
    3. Loop to generate the next token based on previous tokens.
    4. Stop when an end token is generated or max length is reached.
    """

    # ==============================================================================
    # 1. PREPROCESS INPUT (Encoder Input)
    # ==============================================================================
    # TODO: Tokenize the input string.
    # - Instruction: Use 'arabic_tokenizer.texts_to_sequences'.
    #   Note: Pass the input as a list: [input_sequence].
    # - Assign to 'input_seq'.

    # TODO: Pad the input.
    # - Instruction: Use 'pad_sequences'.
    # - Args: 'maxlen=max_seq_length', 'padding="post"'.
    # - Assign to 'padded_input_seq'.


    # ==============================================================================
    # 2. INITIALIZE TARGET (Decoder Input)
    # ==============================================================================
    # We need a container for the result and a buffer to feed into the decoder.

    # TODO: Create a list to hold the predicted token IDs.
    # - Instruction: Initialize an empty list named 'translated_sequence'.

    # TODO: Create the input tensor for the decoder.
    # - Instruction: Create a numpy array of zeros with shape (1, max_seq_length).
    # - Instruction: Set dtype to 'int32'.
    # - Assign to 'target_seq'.


    # ==============================================================================
    # 3. SETUP SPECIAL TOKENS
    # ==============================================================================
    # TODO: Get the IDs for the Start ('#') and End ('$') tokens.
    # - Instruction: Look them up in 'latin_tokenizer.word_index'.
    # - Instruction: Provide default values (e.g., 1 and 2) if not found, just in case.
    # - Assign to 'start_token' and 'end_token'.

    # TODO: Seed the decoder.
    # - Instruction: Set the first element of 'target_seq' (index [0, 0]) to 'start_token'.


    # ==============================================================================
    # 4. GENERATION LOOP
    # ==============================================================================
    # The Transformer outputs a sequence. We look at the output corresponding to the
    # last token we fed in to predict the next one.

    # TODO: Loop from 1 to max_seq_length.
    # - Instruction: Use 'range(1, max_seq_length)'.
    #   (We start at 1 because index 0 is already filled with the start token).
    for i in range(1, max_seq_length):

        # --- A. Predict ---
        # TODO: Get model predictions.
        # - Instruction: Call 'model.predict'.
        # - Inputs: Pass a list containing [padded_input_seq, target_seq].
        # - Args: Set 'verbose=0' to keep the console clean.
        # - Assign result to 'output'.


        # --- B. Select Token ---
        # The model outputs shape (1, max_len, vocab_size).
        # We need the prediction corresponding to the *previous* token (i-1).

        # TODO: Find the most likely token.
        # - Instruction: Select the token at index 'i-1' from the 'output'.
        #   (Slice: output[0, i-1, :]).
        # - Instruction: Use 'np.argmax' on this slice to get the integer ID.
        # - Assign to 'predicted_token_index'.


        # --- C. Check Stopping Conditions ---
        # TODO: Check if we are done.
        # - Instruction: Check if 'predicted_token_index' equals 'end_token'.
        # - Instruction: OR check if 'predicted_token_index' equals 0 (padding).
        # - If either is true, 'break' the loop.


        # --- D. Update ---
        # TODO: Store the result.
        # - Instruction: Append 'predicted_token_index' to the 'translated_sequence' list.

        # TODO: Update the decoder input for the next iteration.
        # - Instruction: Check 'if i < max_seq_length'.
        # - Instruction: Assign 'predicted_token_index' to 'target_seq[0, i]'.


    # ==============================================================================
    # 5. DECODE TO TEXT
    # ==============================================================================
    # TODO: Convert IDs back to words/characters.
    # - Instruction: Use 'latin_tokenizer.sequences_to_texts'.
    #   Pass '[translated_sequence]'.
    # - Instruction: Take the first element [0] of the result.
    # - Assign to 'translated_text'.

    # TODO: Clean up formatting.
    # - Instruction: The tokenizer puts spaces between every token. If this is character-level
    #   generation or specific formatting, remove the spaces using '.replace(" ", "")'.
    # - Update 'translated_text'.

    return translated_text

# Step 6: Test translation with examples
print("\n" + "="*70)
print("TRANSLATION EXAMPLES")
print("="*70)

# Test on custom words
print("\nCustom Arabic Words:")
print("-" * 70)
test_words = ["محمد", "حميد","بولغارد","مادارا"]
for word in test_words:
    translated = translate_sequence(
        "#" + word + "$", transformer_model, arabic_tokenizer, latin_tokenizer, max_seq_length
    )
    print(f"Arabic: {word:<15} -> Transliteration: {translated}")